In [ ]:
# Instala as versões corretas e sincronizadas das bibliotecas
# !pip install -q -U transformers datasets peft trl accelerate bitsandbytes torchao
# !pip install -q -U transformers peft trl accelerate bitsandbytes torchao #torch

!pip install -q -U datasets huggingface_hub transformers peft trl accelerate bitsandbytes torchao

In [ ]:
# !pip install -q "datasets==2.14.0" "huggingface_hub==0.17.3"

In [ ]:
# !pip install -q "datasets==2.14.0"

In [ ]:
# print(datasets.__version__)


In [ ]:
# =======================================================================
# 0. CONFIGURAÇÕES DE MEMÓRIA (DEVE SER A PRIMEIRA COISA DA CÉLULA)
# =======================================================================
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_SKIP_ALLOCATOR_WARMUP"] = "1"

import torch
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer

In [ ]:
# =======================================================================
# PASSO 1 — Modelo e Tokenizador
# =======================================================================
print("=========================================================")
print("[PASSO 1/6] Carregando o BitNet b1.58 na GPU A100...")
print("=========================================================")

model_id = "microsoft/bitnet-b1.58-2B-4T-bf16"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map={"": "cuda:0"},
    attn_implementation="sdpa"
)

[PASSO 1/6] Carregando o BitNet b1.58 na GPU A100...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]

In [ ]:
# =======================================================================
# PASSO 2 — LoRA (idêntico ao SAMSum e MediaSum)
# =======================================================================
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 7,987,200 || all params: 2,420,807,680 || trainable%: 0.3299


In [ ]:
print("=========================================================")
print("[PASSO 3/6] Carregando o QMSum via ZIP...")
print("=========================================================")

from huggingface_hub import hf_hub_download
import zipfile
import json
import os
from datasets import Dataset

# Baixa o ZIP
zip_path = hf_hub_download(
    repo_id="tau/scrolls",
    filename="qmsum.zip",
    repo_type="dataset"
)

# Extrai
extract_path = "./qmsum_data"
os.makedirs(extract_path, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

# Verifica o que foi extraído
for root, dirs, files in os.walk(extract_path):
    for f in files:
        print(os.path.join(root, f))


# Carrega os arquivos JSONL e cria os datasets
dataset = load_dataset("json", data_files={
    "train": "./qmsum_data/qmsum/train.jsonl",
    "validation": "./qmsum_data/qmsum/validation.jsonl"
})

dataset_treino = dataset["train"]
dataset_val = dataset["validation"]

print(f"-> Treino: {len(dataset_treino)} instâncias")
print(f"-> Validação: {len(dataset_val)} instâncias")
print("\nAmostra do primeiro exemplo:")
print(dataset_treino[0])

[PASSO 3/6] Carregando o QMSum via ZIP...
./qmsum_data/__MACOSX/._qmsum
./qmsum_data/__MACOSX/qmsum/._train.jsonl
./qmsum_data/__MACOSX/qmsum/._test.jsonl
./qmsum_data/__MACOSX/qmsum/._.DS_Store
./qmsum_data/__MACOSX/qmsum/._validation.jsonl
./qmsum_data/qmsum/train.jsonl
./qmsum_data/qmsum/validation.jsonl
./qmsum_data/qmsum/.DS_Store
./qmsum_data/qmsum/test.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

-> Treino: 1257 instâncias
-> Validação: 272 instâncias

Amostra do primeiro exemplo:
{'id': 'tr-sq-1', 'pid': 'tr-sq-1_0', 'input': "How Did Project Manager and User Interface introduce the prototype of the remote control?\n\nProject Manager: Yep . Soon as I get this . Okay . This is our last meeting . Um I'll go ahead and go through the minutes from the previous meeting . Uh and then we'll have a , the prototype presentation . {vocalsound} Um then we will um do an evaluation . Uh or we'll see what , what we need to have under the criteria for the evaluation . Then we'll go through the finance and see if we fall within the budget . Um then we'll do the evaluation , and then we can finish up after that with um any changes that we'll need to make , or hopefully everything will fall right in line . Um let's see , minutes from the last meeting . Um we looked at uh the the trends . We had uh the fashion trends that people want a fancy look-and-feel . It was twice as important as anything e

In [ ]:
# =======================================================================
# PASSO 4 — Formatação dos prompts
# =======================================================================
def formata_prompt_qmsum(exemplos):
    if isinstance(exemplos['input'], list):
        textos_finais = []
        for i in range(len(exemplos['input'])):
            texto = f"### Meeting History and Question:\n{exemplos['input'][i]}\n\n### Summary:\n{exemplos['output'][i]}{tokenizer.eos_token}"
            textos_finais.append(texto)
        return textos_finais
    else:
        return f"### Meeting History and Question:\n{exemplos['input']}\n\n### Summary:\n{exemplos['output']}{tokenizer.eos_token}"

In [ ]:
# =======================================================================
# PASSO 5 — Treinamento
# =======================================================================
print("=========================================================")
print("[PASSO 5/6] Iniciando Fine-Tuning QMSum (Contexto Longo)...")
print("=========================================================")

training_args = SFTConfig(
    output_dir="./resultados_qmsum",
    max_length=16384,                     # Corrigido: max_seq_length → max_length (trl 1.5.1)
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=25,
    learning_rate=2e-4,
    num_train_epochs=3,
    bf16=True,
    save_strategy="steps",               # Alterado de "no" para segurança (treino longo)
    save_steps=100,
    save_total_limit=1,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_treino,
    eval_dataset=dataset_val,
    formatting_func=formata_prompt_qmsum,
    args=training_args
    # peft_config removido — LoRA já aplicado no Passo 2
)

trainer.train()

[PASSO 5/6] Iniciando Fine-Tuning QMSum (Contexto Longo)...


Applying formatting function to train dataset:   0%|          | 0/1257 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/1257 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1257 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/272 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/272 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/272 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,2.662725,2.499258,2.343279,4704584.000000,0.465682
100,2.430492,2.380692,2.304057,9393827.000000,0.480460
150,2.389013,2.344389,2.294884,14020894.000000,0.484882
200,2.361268,2.323470,2.287138,18741003.000000,0.487489
250,2.345434,2.309930,2.277345,23340272.000000,0.489178
300,2.351532,2.300690,2.270974,27997846.000000,0.490275
350,2.314953,2.294312,2.263490,32637645.000000,0.491239
400,2.320430,2.290233,2.266608,37190620.000000,0.491753
450,2.315296,2.288335,2.260301,41930860.000000,0.492093
474,2.315296,2.288094,2.260375,44156664.000000,0.492079


TrainOutput(global_step=474, training_loss=2.3974798902680603, metrics={'train_runtime': 11629.8749, 'train_samples_per_second': 0.324, 'train_steps_per_second': 0.041, 'total_flos': 5.543795829812429e+17, 'train_loss': 2.3974798902680603, 'epoch': 3.0})

In [ ]:
# =======================================================================
# PASSO 6 — Salvar modelo, histórico e gráficos
# =======================================================================
pasta_salvamento = "./modelo_final_qmsum"
os.makedirs(pasta_salvamento, exist_ok=True)
trainer.model.save_pretrained(pasta_salvamento)
print(f"Pesos LoRA salvos em: {pasta_salvamento}")

history = pd.DataFrame(trainer.state.log_history)
history.to_csv(f"{pasta_salvamento}/historico_treinamento.csv", index=False)
history.to_json(f"{pasta_salvamento}/historico_treinamento.json", orient="records", indent=4)

train_df = history[history['loss'].notna()]
eval_df = history[history['eval_loss'].notna()]

plt.figure(figsize=(10, 5))
plt.plot(train_df['step'], train_df['loss'], label='Train Loss', color='tab:blue')
if not eval_df.empty:
    plt.plot(eval_df['step'], eval_df['eval_loss'], label='Eval Loss', color='tab:orange')

plt.title('BitNet b1.58 + LoRA (QMSum A100 16k) - Curva de Aprendizado')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.savefig(f"{pasta_salvamento}/loss_curve.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{pasta_salvamento}/loss_curve.pdf", bbox_inches='tight')
plt.close()
print("Gráficos e histórico salvos com sucesso!")

Pesos LoRA salvos em: ./modelo_final_qmsum
Gráficos e histórico salvos com sucesso!


In [ ]:
# =======================================================================
# PASSO 7 — Backup e download
# =======================================================================
import shutil

pasta_backup = "./backup_qmsum_completo"
os.makedirs(pasta_backup, exist_ok=True)

shutil.copytree("./resultados_qmsum", f"{pasta_backup}/resultados_qmsum", dirs_exist_ok=True)
shutil.copytree("./modelo_final_qmsum", f"{pasta_backup}/modelo_final_qmsum", dirs_exist_ok=True)

shutil.make_archive("./backup_qmsum_completo", "zip", pasta_backup)

print(f"Backup gerado: backup_qmsum_completo.zip")
print(f"Tamanho: {os.path.getsize('./backup_qmsum_completo.zip') / (1024**2):.1f} MB")

from google.colab import files
files.download("./backup_qmsum_completo.zip")

Backup gerado: backup_qmsum_completo.zip
Tamanho: 72.9 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>